In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D, BatchNormalization, Activation, AveragePooling1D, LSTM, Add, Dense
from tensorflow.keras.models import Model

# 1. Config parameters
history_lengths = 44#[44, 92, 182]
pooling_widths  = 7#[7, 15, 30]
conv_filters    = 2#[2, 2, 2]
conv_widths     = 3#[3, 3, 3]
embedding_dims  = 32
lstm_units      = 16
hidden_neurons  = 8#[8, 8]
conv_act        = 'relu'
hidden_act      = 'relu'
VOCAB_SIZE      = 128

# 2. Build a slice
def build_slice(hist_len, pool_width, conv_filter, conv_width, tag):
    inp = Input(shape=(hist_len,), dtype='int32', name=f'{tag}_in')
    x = Embedding(VOCAB_SIZE, embedding_dims, name=f'{tag}_emb')(inp)
    x = Conv1D(conv_filter, conv_width, padding='valid', name=f'{tag}_conv')(x)
    x = BatchNormalization(name=f'{tag}_bn')(x)
    x = Activation(conv_act, name=f'{tag}_act')(x)
    x = AveragePooling1D(pool_width, name=f'{tag}_pool')(x)
    x = LSTM(lstm_units, name=f'{tag}_lstm')(x)
    x = BatchNormalization(name=f'{tag}_lstm_bn')(x)
    x = Activation('tanh', name=f'{tag}_tanh')(x)
    return inp, x

# 3. Build all slices
# inputs, vectors = zip(*[
#     build_slice(history_lengths[i], pooling_widths[i], conv_filters[i], conv_widths[i], f'slice{i}')
#     for i in range(1)
# ])

inputs, vectors = zip(*[
    build_slice(history_lengths, pooling_widths, conv_filters, conv_widths, f'slice{1}')])

# 4. Folded Add (for hls4ml compatibility)
merged = vectors[0]
for i, v in enumerate(vectors[1:], start=1):
    merged = Add(name=f'add_fold_{i}')([merged, v])

# 5. FC Head
x = merged
# for i, n in enumerate(hidden_neurons):
#     x = Dense(n, name=f'fc_{i}')(x)
#     x = BatchNormalization(name=f'fc_{i}_bn')(x)
#     x = Activation(hidden_act, name=f'fc_{i}_act')(x)

x = Dense(hidden_neurons, name='fc_0')(x)
x = BatchNormalization(name='fc_0_bn')(x)
x = Activation(hidden_act, name='fc_0_act')(x)

output = Dense(1, activation='sigmoid', name='output')(x)

model = Model(inputs=list(inputs), outputs=output, name='BranchNet_Keras_Modular')
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


2025-08-13 11:37:38.754642: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-13 11:37:38.818662: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-13 11:37:39.138401: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-13 11:37:39.138508: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-13 11:37:39.194616: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Model: "BranchNet_Keras_Modular"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 slice1_in (InputLayer)      [(None, 44)]              0         
                                                                 
 slice1_emb (Embedding)      (None, 44, 32)            4096      
                                                                 
 slice1_conv (Conv1D)        (None, 42, 2)             194       
                                                                 
 slice1_bn (BatchNormalizat  (None, 42, 2)             8         
 ion)                                                            
                                                                 
 slice1_act (Activation)     (None, 42, 2)             0         
                                                                 
 slice1_pool (AveragePoolin  (None, 6, 2)              0         
 g1D)                                      

In [5]:
import tensorflow as tf
import yaml
from dataloader_tf import BranchTraceDatasetTFSingle

# Load config file
with open('KladosNet/branchnet/configs/mini_250.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# Parameters for loader
trace_paths = ['../traces/641.leela_s-602B_dataset.hdf5']
br_pc = 4320802
history_length = history_lengths  # Now it's a single value, not a list
pc_bits = 11
pc_hash_bits = 6
hash_dir_with_pc = True

# Create Dataset
dataset_loader = BranchTraceDatasetTFSingle(
    trace_paths=trace_paths,
    br_pc=br_pc,
    history_length=history_length,  # Provide a list here, even if it's a single value
    pc_bits=pc_bits,
    pc_hash_bits=pc_hash_bits,
    hash_dir_with_pc=hash_dir_with_pc
)
dataset = dataset_loader.get_dataset()#batch_size=128
model.fit(dataset, epochs=1)

    111/Unknown - 30s 199ms/step - loss: 0.3383 - accuracy: 0.9197

KeyboardInterrupt: 

In [6]:
# Suppose you have test_traces, test_br_pc, etc.
test_trace_paths = ['../traces/641.leela_s-862B_dataset.hdf5']

test_dataset_loader = BranchTraceDatasetTFSingle(
    trace_paths=test_trace_paths,
    br_pc=4320802,  # use same branch PC or new one as needed
    history_length=history_lengths, 
    pc_bits=pc_bits,
    pc_hash_bits=pc_hash_bits,
    hash_dir_with_pc=hash_dir_with_pc
)
test_dataset = test_dataset_loader.get_dataset()  # batch_size=128

results = model.evaluate(test_dataset)
print('Test loss, Test accuracy:', results)

    103/Unknown - 22s 180ms/step - loss: 0.6327 - accuracy: 0.9380

KeyboardInterrupt: 

In [ ]:
import hls4ml 
import os
import pathlib

os.environ['PATH'] = '/tools/Xilinx/Vivado/2019.1/bin' + os.pathsep + os.environ['PATH']

cfg = hls4ml.utils.config_from_keras_model(model, granularity='name')
cfg['Model'].update({
    'Strategy': 'Resource',
    'IOType': 'io_stream',
    'ReuseFactor': 128,                # modest global RF
    'Precision': 'fixed<16,6>',      # or 'fixed<12,4>' after checking accuracy
})

# Convs: stream-friendly implementation, no extra parallelization
lt = cfg.setdefault('LayerType', {})
lt.setdefault('Conv2D', {})['ConvImplementation'] = 'linebuffer'
lt['Conv2D']['ParallelizationFactor'] = 1

# Override only the heavy layers with valid reuse divisors from the build log
cfg['LayerName'].setdefault('out', {}).update({'Strategy':'Resource', 'ReuseFactor': 128})

# If you can use the Vivado backend, enable FIFO depth optimization:
# cfg['Flows'] = ['vivado:fifo_depth_optimization']
# hls4ml.model.optimizer.get_optimizer('vivado:fifo_depth_optimization') \
#     .configure(profiling_fifo_depth=100_000)

hls_model = hls4ml.converters.convert_from_keras_model(
    model, hls_config=cfg, backend='VivadoAccelerator',
    board='pynq-z2', clock_period=10, output_dir='BranchNet_Accel_clean'
)

hls_model.build(csim=False, synth=True, export=True, bitfile=True)

WARN: Unable to import optimizer(s) from expr_templates.py: No module named 'sympy'
Interpreting Model
Topology:
Layer name: slice1_in, layer type: InputLayer, input shapes: [[None, 44]], output shape: [None, 44]
Layer name: slice1_emb, layer type: Embedding, input shapes: [[None, 44]], output shape: [None, 44, 32]
Layer name: slice1_conv, layer type: Conv1D, input shapes: [[None, 44, 32]], output shape: [None, 42, 2]
Layer name: slice1_bn, layer type: BatchNormalization, input shapes: [[None, 42, 2]], output shape: [None, 42, 2]
Layer name: slice1_act, layer type: Activation, input shapes: [[None, 42, 2]], output shape: [None, 42, 2]
Layer name: slice1_pool, layer type: AveragePooling1D, input shapes: [[None, 42, 2]], output shape: [None, 6, 2]
Layer name: slice1_lstm, layer type: LSTM, input shapes: [[None, 6, 2]], output shape: [None, 16]
Layer name: slice1_lstm_bn, layer type: BatchNormalization, input shapes: [[None, 16]], output shape: [None, 16]
Layer name: slice1_tanh, layer ty